In [ ]:
# =============================================================================
# nb53 - RECOMPUTE THE MECHANISM CORRELATION WITH U2R EXCLUDED
#
# WHY. Section 4.11 states that a class whose calibration count falls below
# ceil(1/alpha) - 1 has no formable quantile and is excluded from analysis rather
# than counted as covered. NSL-KDD U2R has seven source calibration points against
# a floor of nineteen, yet it appears in the pooled mechanism table with a FINITE
# quantile (q_src around 0.977) and non-trivial coverage. The pooled statistic was
# therefore computed over 28 cells when the stated rule admits 27.
#
# This notebook recomputes the class-level statistic with U2R excluded and commits
# the result, so the manuscript's figures derive from a committed file rather than
# from an ad hoc recalculation.
#
# The exclusion is conservative: it lowers the reported correlation.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
import os, sys, json, shutil, subprocess, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
FLOOR=int(np.ceil(1/ALPHA))-1
print(f'alpha {ALPHA} | feasibility floor {FLOOR} calibration points')


In [ ]:
# =============================================================================
# Cell 2 - rebuild the class-level table and audit feasibility per cell.
# =============================================================================
d=pd.read_csv(RD/'score_shift_explainability.csv')
print(f'arch-level rows: {len(d)}')
cl=d.groupby(['dataset','class'],as_index=False).agg(
    n_src=('n_src','mean'), n_tgt=('n_tgt','mean'),
    ks=('score_KS','mean'), coverage=('SHC_coverage','mean'),
    under=('undercoverage','mean'), q_src=('q_src','mean'))
cl['env']=cl.dataset.str.split(':').str[0]
cl['feasible']=cl.n_src >= FLOOR
cl['included']=cl.feasible

print(f'\nclass-level cells: {len(cl)}')
print('\nFEASIBILITY AUDIT — every cell, so the accounting is explicit')
print(cl[['env','class','n_src','q_src','coverage','under','ks','feasible','included']]
      .round(4).sort_values(['env','class']).to_string(index=False))
excl=cl[~cl.feasible]
print(f'\ncells below the floor of {FLOOR}: {len(excl)}')
if len(excl):
    print(excl[['env','class','n_src','q_src','coverage']].round(4).to_string(index=False))
    print('\n  Note the finite q_src on an infeasible cell. That is the inconsistency:')
    print('  the quantile was formed despite the count being below the floor.')


In [ ]:
# =============================================================================
# Cell 3 - the statistic, with and without the infeasible cells.
# =============================================================================
def spear(g):
    r,p=stats.spearmanr(g.ks, g.under); return float(r), float(p)
rng=np.random.default_rng(20260730)
def boot(x,y,B=4000):
    o=[]
    for _ in range(B):
        i=rng.choice(len(x),len(x),replace=True)
        if len(np.unique(x[i]))<3: continue
        o.append(stats.spearmanr(x[i],y[i])[0])
    return float(np.percentile(o,2.5)), float(np.percentile(o,97.5))

inc=cl[cl.included].reset_index(drop=True)
r_inc,p_inc=spear(inc); lo,hi=boot(inc.ks.to_numpy(), inc.under.to_numpy())
r_all,p_all=spear(cl);  lo_a,hi_a=boot(cl.ks.to_numpy(), cl.under.to_numpy())

print('POOLED CLASS-LEVEL STATISTIC')
print(f'  included only (n={len(inc)}): rho {r_inc:+.4f}  p {p_inc:.3e}  95% CI [{lo:.3f}, {hi:.3f}]')
print(f'  all cells     (n={len(cl)}): rho {r_all:+.4f}  p {p_all:.3e}  95% CI [{lo_a:.3f}, {hi_a:.3f}]')
print(f'  effect of excluding infeasible cells: {r_inc-r_all:+.4f}  '
      f"({'conservative' if r_inc<r_all else 'anti-conservative'})")

print('\nPER DATASET (included cells only)')
per={}
for e,g in inc.groupby('env'):
    if len(g)>2:
        r,p=spear(g); per[e]={'rho':round(r,4),'p':round(p,4),'n':int(len(g))}
        print(f'  {e:12s} n={len(g)} rho {r:+.3f} p {p:.4f}')

print('\nLEAVE-ONE-DATASET-OUT (included cells only)')
loo={}
for e in sorted(inc.env.unique()):
    g=inc[inc.env!=e]
    r,_=spear(g); loo[e]=round(r,4)
    print(f'  drop {e:12s} n={len(g):2d} rho {r:+.3f}')


In [ ]:
# =============================================================================
# Cell 4 - commit the ledger row so the manuscript figures have provenance.
# =============================================================================
cl.to_csv(RD/'mechanism_class_level_feasibility_audit.csv', index=False)
out={
 'rule':f'a class with fewer than {FLOOR} source calibration points has no formable '
        f'quantile at alpha={ALPHA} and is excluded (Section 4.11)',
 'n_cells_total':int(len(cl)),
 'n_cells_included':int(len(inc)),
 'excluded':[{'env':r.env,'class':r['class'],'n_src':float(r.n_src),'q_src':float(r.q_src)}
             for _,r in cl[~cl.feasible].iterrows()],
 'class_level_spearman':round(r_inc,4),
 'p':p_inc,
 'ci95':[round(lo,3),round(hi,3)],
 'with_infeasible_included':{'rho':round(r_all,4),'n':int(len(cl)),
                             'ci95':[round(lo_a,3),round(hi_a,3)]},
 'per_dataset':per,
 'leave_one_dataset_out':loo,
 'supersedes':'mechanism_pooled_four_datasets.json, which reported rho 0.8439 over 28 cells '
              'with NSL-KDD U2R included despite its infeasibility',
}
(RD/'mechanism_pooled_feasible_only.json').write_text(json.dumps(out, indent=2, default=str))
print('saved mechanism_pooled_feasible_only.json and the per-cell audit')
print(json.dumps({k:v for k,v in out.items() if k not in ('excluded','per_dataset','leave_one_dataset_out')},
                 indent=2, default=str))

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','nb53: exclude infeasible NSL-KDD U2R from the mechanism statistic per the Section 4.11 rule; rho 0.844 (n=28) -> 0.828 (n=27)')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
